Recurrent Neural Network(RNN), Long-Short Term Memory(LSTM) and Gated Recurrent Unit(GRU) for Time Series Forecasting

1. Load the time series dataset
2. Forecasting using RNNs

We can implement the above problem using build-in-methods defined in keras.

In [ ]:
# 0. Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import sklearn.preprocessing
from sklearn.metrics import r2_score
from keras.layers import Dense,Dropout,SimpleRNN,LSTM
from keras.models import Sequential

In [ ]:
file_path = '.data/dom_hourly.csv'
data = pd.read_csv(file_path, index_col='Datetime, parse_dates=['Datetime'])
data.head()

In [ ]:
# check the missing data
data.isna().sum()

In [ ]:
# time series data
data.plot(figsize=(16, 4), legend=True)
plt.title('DOM hourly power consumption data - Before normalization)
plt.show()

In [ ]:
# function to normalize the data
def normalize_fn(data):
    scaler_object = sklearn.preprocessing.MinMaxScaler()
     data['DOM_MW']=scaler_object.fit_transform(data['DOM_MW'].
values.reshape(-1,1))
    return data

data_norm = normalize_fn(data)
data_norm.head()

In [ ]:
data_norm.plot(figsize=(16,4),legend=True)
plt.title('DOM hourly power consumption data - AFTER
NORMALIZATION')
plt.show()

In [ ]:
def data_prep(data, length):
    X = []
    y = []
    for i in range(length, len(data)):
        X.append(data.iloc[i - length: i, 0])
        y.append(data.iloc[i, 0])
    # train-test split
     # training contains first 110000 days and test contains the
remaining 6189 days
    train_X = X[:110000]
    train_y = y[:110000]
    test_X = X[110000:]
    test_y = y[110000:]
    # converting to numpy array
    train_X = np.array(train_X)
    train_y = np.array(train_y)
    test_X = np.array(test_X)
    test_y = np.array(test_y)
     # reshaping data to required format to input to RNN,
LSTM models
    train_X = np.reshape(train_X, (110000, length, 1))
    test_X = np.reshape(test_X, (test_X.shape[0], length, 1))
    return [train_X, train_y, test_X, test_y]

In [ ]:
sequence_length = 20
train_X, train_y, test_X, test_y = data_prep(data,
sequence_length)
print('train_X.shape = ',train_X.shape)
print('train_y.shape = ', train_y.shape)
print('test_X.shape = ', test_X.shape)
print('test_y.shape = ',test_y.shape)

In [ ]:
# 1. Initialize the rnn model
model = Sequential()
model.add(LSTM(40, activation="tanh", return_sequences=True, input_shape = (train_X.shape[1], 1)))
model.add(Dropout(0.15))
model.add(LSTM(40, activation="tanh", return_sequences=True))
model.add(Dropout(0.15))
model.add(LSTM(40, activation="tanh", return_sequences=False))
model.add(Dropout(0.15))
model.add(Dense(1))

In [ ]:
# 2. Create model summary
model.summary()

In [ ]:
# 3. fit the model
model.compile(optimizer="adam", loss="MSE")
model.fit(train_X, train_y, epochs=10, batch_size=1000)

In [ ]:
# 4. make model predictions and print score
predictions = model.predict(test_X)
score = r2_score(test_y, predictions)
print("R-Square Score of RNN model = ", score)

In [ ]:
# 5. Visualization of plot of the function of actual vs pred function
plotting_actual_vs_pred(test_y, predictions, "Predictions made by RNN model")